In [56]:
using Turing, StatsPlots, Distributions, CSV, DataFrames, Optim, Random

# Load the observed dataset
Random.seed!(42)
data = CSV.read("../../data/ice_cream_sales.csv", DataFrame);
names(data)

6-element Vector{String}:
 "ID"
 "Temperature"
 "Is_Weekend"
 "Hours_Open"
 "Electricity_Usage"
 "Ice_Cream_Sales"

In [57]:
describe(data)

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Float64,Real,Float64,Real,Int64,DataType
1,ID,180.5,1,180.5,360,0,Int64
2,Temperature,24.7859,19.2983,24.8258,29.7834,0,Float64
3,Is_Weekend,0.283333,false,0.0,true,0,Bool
4,Hours_Open,9.40833,8,10.0,11,0,Int64
5,Electricity_Usage,88.1732,65.6504,87.9184,115.253,0,Float64
6,Ice_Cream_Sales,1028.59,803,1025.5,1273,0,Int64


In [58]:
#= Convert continous data to Float64
And the categorical data to integers =#

data[!, :Temperature] = convert(Array{Float64}, data[!, :Temperature])
data[!, :Is_Weekend] = convert(Array{Int}, data[!, :Is_Weekend])
data[!, :Hours_Open] = convert(Array{Int}, data[!, :Hours_Open])
data[!, :Ice_Cream_Sales] = convert(Array{Float64}, data[!, :Ice_Cream_Sales])
data[!, :Electricity_Usage] = convert(Array{Float64}, data[!, :Electricity_Usage])
temperature = data[!, :Temperature]
is_weekend = data[!, :Is_Weekend]
hours_open = data[!, :Hours_Open]
ice_cream_sales = data[!, :Ice_Cream_Sales]
electricity_usage = data[!, :Electricity_Usage];

In [59]:
# Build the model

@model function ice_cream_sales_model(temperature, is_weekend, hours_open, electricity_usage, ice_cream_sales)
    n = min(length(temperature), length(is_weekend), length(hours_open), length(electricity_usage), length(ice_cream_sales))
    
    for i in 1:n
        # Priors for independent variables
        temperature[i] ~ Normal(25, 2)
        is_weekend[i] ~ Bernoulli(2/7)

        # How each model is dependent on it's predecessors
        if is_weekend[i] == 1
            hours_open[i] ~ DiscreteUniform(10,11)
        else
            hours_open[i] ~ DiscreteUniform(8,10)
        end

        electricity_usage[i] ~ Normal(10 + temperature[i] * 2 + hours_open[i] * 3, 1)
        
        # How the dependent variable is dependent on the independent variables
        ice_cream_sales[i] ~ Normal(20 + temperature[i] * 30 + hours_open[i] * 25 + is_weekend[i] * 100, 3)
    end
end

ice_cream_sales_model (generic function with 2 methods)

In [60]:
cont_sampler = HMC(0.01, 10, :temperature, :electricity_usage, :ice_cream_sales)
disc_sampler = PG(10, :is_weekend, :hours_open)
sampler = Gibbs(cont_sampler, disc_sampler)

Gibbs{(:temperature, :electricity_usage, :ice_cream_sales, :is_weekend, :hours_open), Tuple{HMC{Turing.Essential.ForwardDiffAD{0}, (:temperature, :electricity_usage, :ice_cream_sales), AdvancedHMC.UnitEuclideanMetric}, PG{(:is_weekend, :hours_open), AdvancedPS.ResampleWithESSThreshold{typeof(AdvancedPS.resample_systematic), Float64}}}}((HMC{Turing.Essential.ForwardDiffAD{0}, (:temperature, :electricity_usage, :ice_cream_sales), AdvancedHMC.UnitEuclideanMetric}(0.01, 10), PG{(:is_weekend, :hours_open), AdvancedPS.ResampleWithESSThreshold{typeof(AdvancedPS.resample_systematic), Float64}}(10, AdvancedPS.ResampleWithESSThreshold{typeof(AdvancedPS.resample_systematic), Float64}(AdvancedPS.resample_systematic, 0.5))))

In [61]:
#model = ice_cream_sales_model(temperature, is_weekend, hours_open, electricity_usage, repeat([missing], length(temperature)))

In [62]:
#chain = sample(model, sampler, MCMCThreads(), 500, 6)

In [63]:
model_intervention = ice_cream_sales_model(temperature, is_weekend, hours_open, electricity_usage .+ 20, repeat([missing], length(temperature)))

DynamicPPL.Model{typeof(ice_cream_sales_model), (:temperature, :is_weekend, :hours_open, :electricity_usage, :ice_cream_sales), (), (), Tuple{Vector{Float64}, Vector{Int64}, Vector{Int64}, Vector{Float64}, Vector{Missing}}, Tuple{}, DynamicPPL.DefaultContext}(ice_cream_sales_model, (temperature = [24.273285037096446, 25.503474431148458, 24.37002405766209, 24.37749519735116, 26.632613529864656, 25.953476759663754, 23.280889235876757, 22.06142358898691, 20.77133033773803, 25.08756332406122  …  27.88968302198113, 22.020745012649975, 24.783201090136597, 21.12625085079417, 24.805501713106118, 20.97052730229581, 27.764905318715257, 21.868932876733844, 22.15767543558101, 24.99857631463805], is_weekend = [0, 0, 0, 0, 0, 1, 1, 0, 0, 0  …  0, 0, 0, 0, 0, 1, 1, 0, 0, 0], hours_open = [10, 9, 8, 10, 8, 11, 11, 8, 9, 10  …  9, 8, 10, 10, 9, 11, 11, 9, 8, 10], electricity_usage = [112.11831434065243, 103.91781748696985, 108.29061662682687, 105.5618654162385, 114.404976320194, 118.23792371386881, 106

In [64]:
chain_intervention = sample(model_intervention, sampler, MCMCThreads(), 500, 6)

Sampling (6 threads)   0%|                              |  ETA: N/A
Sampling (6 threads)  17%|█████                         |  ETA: 0:20:50
Sampling (6 threads)  33%|██████████                    |  ETA: 0:08:20
Sampling (6 threads)  50%|███████████████               |  ETA: 0:04:10
Sampling (6 threads)  67%|████████████████████          |  ETA: 0:02:05
Sampling (6 threads)  83%|█████████████████████████     |  ETA: 0:00:50
Sampling (6 threads) 100%|██████████████████████████████| Time: 0:13:57
Sampling (6 threads) 100%|██████████████████████████████| Time: 0:13:57


Chains MCMC chain (500×361×6 Array{Float64, 3}):

Iterations        = 1:1:500
Number of chains  = 6
Samples per chain = 500
Wall duration     = 836.03 seconds
Compute duration  = 2080.02 seconds
parameters        = ice_cream_sales[1], ice_cream_sales[2], ice_cream_sales[3], ice_cream_sales[4], ice_cream_sales[5], ice_cream_sales[6], ice_cream_sales[7], ice_cream_sales[8], ice_cream_sales[9], ice_cream_sales[10], ice_cream_sales[11], ice_cream_sales[12], ice_cream_sales[13], ice_cream_sales[14], ice_cream_sales[15], ice_cream_sales[16], ice_cream_sales[17], ice_cream_sales[18], ice_cream_sales[19], ice_cream_sales[20], ice_cream_sales[21], ice_cream_sales[22], ice_cream_sales[23], ice_cream_sales[24], ice_cream_sales[25], ice_cream_sales[26], ice_cream_sales[27], ice_cream_sales[28], ice_cream_sales[29], ice_cream_sales[30], ice_cream_sales[31], ice_cream_sales[32], ice_cream_sales[33], ice_cream_sales[34], ice_cream_sales[35], ice_cream_sales[36], ice_cream_sales[37], ice_cream_sales[3

In [65]:
model_counterfacutal = ice_cream_sales_model(repeat([30], length(temperature)), is_weekend, hours_open, electricity_usage, repeat([missing], length(temperature)))

DynamicPPL.Model{typeof(ice_cream_sales_model), (:temperature, :is_weekend, :hours_open, :electricity_usage, :ice_cream_sales), (), (), Tuple{Vector{Int64}, Vector{Int64}, Vector{Int64}, Vector{Float64}, Vector{Missing}}, Tuple{}, DynamicPPL.DefaultContext}(ice_cream_sales_model, (temperature = [30, 30, 30, 30, 30, 30, 30, 30, 30, 30  …  30, 30, 30, 30, 30, 30, 30, 30, 30, 30], is_weekend = [0, 0, 0, 0, 0, 1, 1, 0, 0, 0  …  0, 0, 0, 0, 0, 1, 1, 0, 0, 0], hours_open = [10, 9, 8, 10, 8, 11, 11, 8, 9, 10  …  9, 8, 10, 10, 9, 11, 11, 9, 8, 10], electricity_usage = [92.11831434065243, 83.91781748696985, 88.29061662682687, 85.5618654162385, 94.404976320194, 98.23792371386881, 86.5919633338461, 85.73695548415236, 84.63928092676501, 83.61593564313691  …  99.56271099890587, 76.4706206145493, 89.55295149298311, 72.2699582788947, 80.88052787405555, 103.17739012406682, 92.79240101984547, 89.82611366797212, 82.53449637881273, 97.7094309458657], ice_cream_sales = [missing, missing, missing, missing,

In [66]:
chain_couneterfactual = sample(model_counterfacutal, sampler, MCMCThreads(), 500, 6)

Sampling (6 threads)   0%|                              |  ETA: N/A
Sampling (6 threads)  17%|█████                         |  ETA: 0:21:38
Sampling (6 threads)  33%|██████████                    |  ETA: 0:08:39
Sampling (6 threads)  50%|███████████████               |  ETA: 0:04:20
Sampling (6 threads)  67%|████████████████████          |  ETA: 0:02:10
Sampling (6 threads)  83%|█████████████████████████     |  ETA: 0:00:52
Sampling (6 threads) 100%|██████████████████████████████| Time: 0:14:04
Sampling (6 threads) 100%|██████████████████████████████| Time: 0:14:04


Chains MCMC chain (500×361×6 Array{Float64, 3}):

Iterations        = 1:1:500
Number of chains  = 6
Samples per chain = 500
Wall duration     = 842.81 seconds
Compute duration  = 2135.48 seconds
parameters        = ice_cream_sales[1], ice_cream_sales[2], ice_cream_sales[3], ice_cream_sales[4], ice_cream_sales[5], ice_cream_sales[6], ice_cream_sales[7], ice_cream_sales[8], ice_cream_sales[9], ice_cream_sales[10], ice_cream_sales[11], ice_cream_sales[12], ice_cream_sales[13], ice_cream_sales[14], ice_cream_sales[15], ice_cream_sales[16], ice_cream_sales[17], ice_cream_sales[18], ice_cream_sales[19], ice_cream_sales[20], ice_cream_sales[21], ice_cream_sales[22], ice_cream_sales[23], ice_cream_sales[24], ice_cream_sales[25], ice_cream_sales[26], ice_cream_sales[27], ice_cream_sales[28], ice_cream_sales[29], ice_cream_sales[30], ice_cream_sales[31], ice_cream_sales[32], ice_cream_sales[33], ice_cream_sales[34], ice_cream_sales[35], ice_cream_sales[36], ice_cream_sales[37], ice_cream_sales[3

In [67]:
function calculate_ate_intervention(chain, chain_2, ics)
    ate = 0
    for i in 1:length(ics)
        ate += mean(chain["ice_cream_sales[$i]"]) - mean(chain_2["ice_cream_sales[$i]"])
    end
    ate /= length(ics)
    return ate
end

function calculate_ate_counterfactual(chain, chain_2, ics, temperature)
    ate = 0
    for i in 1:length(ics)
        difference =  mean(chain["ice_cream_sales[$i]"]) - mean(chain_2["ice_cream_sales[$i]"])
        difference /= abs(30 - temperature[i])
        ate += difference
    end
    ate /= length(ics)
    return ate
end

calculate_ate_counterfactual (generic function with 1 method)

In [68]:
function calculate_mse(chain, ice_cream_sales)
    mse = 0
    for i in 1:length(ice_cream_sales)
        difference = mean(chain["ice_cream_sales[$i]"]) - ice_cream_sales[i]
        difference = difference^2
        mse += difference
    end
    mse = mse / length(ice_cream_sales)
    return mse
end

calculate_mse (generic function with 1 method)

In [69]:
sqrt(calculate_mse(chain_intervention, ice_cream_sales))

19.44208236794597

In [70]:
ics_counterfactual = ice_cream_sales .+ (30 .- temperature) .* 30

sqrt(calculate_mse(chain_couneterfactual, ics_counterfactual))

19.349605853362885

In [71]:
calculate_ate_intervention(chain_intervention, chain, ice_cream_sales)

0.04573676337594274

In [72]:
calculate_ate_counterfactual(chain_couneterfactual, chain, ice_cream_sales, temperature)

30.031927782558174